# 13 - Segundo trabalho
Em dupla ou trio, faça de acordo com a especificação do trabalho, usando os conceitos de interpolação, regressão, derivação e integração numérica.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana13
```

Faça o checkout nessa nova branch:

```bash
git checkout semana13
```

<hr />

## Atividade 1
Uma fábrica de laticínios, na etapa de resfriamento para fermentação dos produtos, utiliza diferentes tamanhos de serpentinas para alcançar as temperaturas ideais no processo de fabricação.  

De acordo com o manual do fabricante, as serpentinas seguem precisamente a fórmula:

$$
f(x) = a_3 x^3 + a_2 x^2 + a_1 x + a_0
$$

A tabela contém as temperaturas na primeira linha, os metros de serpentina na segunda, e o tamanho das novas serpentinas na terceira.  
Encontre as temperaturas alcançadas para essas novas serpentinas.

| Valores |
|----------|
| 4.9, 3.3, 3.0, 2.0 |
| 1.5, 2.0, 2.2, 3.0 |
| 1.75, 2.5, 2.75, 3.2 |
| **Resposta aproximada** |
| **3.91, 2.7, 2.43, 1.28** |

**Sugestão:** usar o método da **interpolação polinomial**.

In [1]:
import numpy as np

def polinomial(x, y):
    n = len(x)
    V = np.vander(x, N=n, increasing=False)
    return np.linalg.solve(V, y.astype(float))

def main():
    # x = metros de serpentina, y = temperatura alcançada (°C)
    x = np.array([1.5, 2.0, 2.2, 3.0], dtype=float)
    y = np.array([4.9, 3.3, 3.0, 2.0], dtype=float)

    # Novas serpentinas
    x_novas = np.array([1.75, 2.5, 2.75, 3.2], dtype=float)

    coefs = polinomial(x, y)
    y_novas = np.polyval(coefs, x_novas)

    print("Polinômio interpolador f(x) = a3*x^3 + a2*x^2 + a1*x + a0:")
    print(f"Coeficientes (grau 3 até 0): {coefs}")
    print("\nTemperaturas alcançadas para as novas serpentinas:")
    for l, temp in zip(x_novas, y_novas):
        print(f"Comprimento {l:.2f} m -> Temperatura: {temp:.2f} °C (exato: {temp:.6f})")

if __name__ == "__main__":
    main()

Polinômio interpolador f(x) = a3*x^3 + a2*x^2 + a1*x + a0:
Coeficientes (grau 3 até 0): [ -1.45238095  10.70714286 -27.24047619  26.57142857]

Temperaturas alcançadas para as novas serpentinas:
Comprimento 1.75 m -> Temperatura: 3.91 °C (exato: 3.907366)
Comprimento 2.50 m -> Temperatura: 2.70 °C (exato: 2.696429)
Comprimento 2.75 m -> Temperatura: 2.43 °C (exato: 2.427902)
Comprimento 3.20 m -> Temperatura: 1.45 °C (exato: 1.451429)


## Atividade 2
Suponha um circuito RL em série com uma fonte de tensão constante $V$, resistência $R$ e indutância $L$.  
A corrente $I(t)$ é medida nos seguintes instantes:

| Tempo $t$ (s) | Corrente $I(t)$ (A) |
|----------------|---------------------|
| 0.2 | 0.8187 |
| 0.5 | 1.5815 |
| 0.7 | 1.9354 |
| 1.0 | 2.2905 |

Estime $I(0.3)$ e $I(0.9)$.

**Sugestão:** método da **interpolação polinomial**.  
**Resposta:**  
$I(0.3) = 1.107639\,A$  
$I(0.9) = 2.19182\,A$


In [2]:
import numpy as np

def polinomial(x, y):
    n = len(x)
    V = np.vander(x, N=n, increasing=False)
    return np.linalg.solve(V, y.astype(float))

def main():
    t = np.array([0.2, 0.5, 0.7, 1.0], dtype=float)
    I = np.array([0.8187, 1.5815, 1.9354, 2.2905], dtype=float)

    coefs = polinomial(t, I)
    
    t_est = np.array([0.3, 0.9], dtype=float)
    I_est = np.polyval(coefs, t_est)

    print("Polinômio interpolador da corrente I(t):")
    print(f"Coeficientes: {coefs}")
    print(f"\nI(0.3) = {I_est[0]:.6f} A")
    print(f"I(0.9) = {I_est[1]:.5f} A")

if __name__ == "__main__":
    main()

Polinômio interpolador da corrente I(t):
Coeficientes: [ 0.46833333 -2.202       3.90141667  0.12275   ]

I(0.3) = 1.107640 A
I(0.9) = 2.19182 A


## Atividade 3
Uma fábrica de chuveiros possui um modelo com quatro modos de temperatura: verão, outono, inverno e primavera.  

Deseja-se obter o modelo linear:

$$
f(x) = a_1 x + a_0
$$

onde $x$ é o comprimento da resistência (cm) e $f(x)$ é a temperatura (°C).

### Dados experimentais

| Modelo | Medições (°C) | Resistência (cm) | Resposta aproximada $(a_1, a_0)$ |
|---------|----------------|------------------|-----------------------------------|
| **Outono** | 38, 40, 42, 44, 46 | 1, 1.25, 1.5, 1.75, 2 | (30, 8) |
| **Inverno** | 40, 44, 48, 52, 56 | 1, 1.5, 2, 2.5, 3 | (32, 8) |
| **Primavera** | 36, 39, 42, 45, 48 | 1, 1.15, 1.3, 1.45, 1.6 | (16, 20) |

**Sugestão:** usar o método dos **quadrados mínimos**.

In [3]:
import numpy as np

def regressao(x, y, v):
    V = v(x)
    Vt = V.T
    return np.linalg.inv(Vt @ V) @ (Vt @ y)

def main():
    modelos = {
        "Outono": {
            "T": np.array([38, 40, 42, 44, 46], dtype=float),
            "L": np.array([1, 1.25, 1.5, 1.75, 2], dtype=float),
        },
        "Inverno": {
            "T": np.array([40, 44, 48, 52, 56], dtype=float),
            "L": np.array([1, 1.5, 2, 2.5, 3], dtype=float),
        },
        "Primavera": {
            "T": np.array([36, 39, 42, 45, 48], dtype=float),
            "L": np.array([1, 1.15, 1.3, 1.45, 1.6], dtype=float),
        },
    }

    v = lambda x: np.column_stack((np.ones(len(x)), x))

    print(f"{'Modelo':<12} | {'a0 (intercepto)':>15} | {'a1 (inclinação)':>15} | {'(a0, a1)':>12}")
    print("-" * 60)
    for nome, dados in modelos.items():
        L = dados["L"]
        T = dados["T"]
        A = regressao(L, T, v)
        a0, a1 = A[0], A[1]
        print(f"{nome:<12} | {a0:>15.4f} | {a1:>15.4f} | ({int(round(a0))}, {int(round(a1))})")

if __name__ == "__main__":
    main()

Modelo       | a0 (intercepto) | a1 (inclinação) |     (a0, a1)
------------------------------------------------------------
Outono       |         30.0000 |          8.0000 | (30, 8)
Inverno      |         32.0000 |          8.0000 | (32, 8)
Primavera    |         16.0000 |         20.0000 | (16, 20)


## Atividade 4

Um conversor DC-DC foi testado em várias condições de carga.  
A relação entre tensão e corrente é:

$$
V = R_{eq} I + b
$$

| Tensão $V$ (V) | Corrente $I$ (A) |
|----------------|------------------|
| 5.1 | 0.5 |
| 10.3 | 1.1 |
| 15.2 | 1.5 |
| 20.1 | 2.2 |
| 24.7 | 2.5 |
| 30.5 | 3.1 |

**Sugestão:** método dos **quadrados mínimos**.  
**Resposta:**  
$R_{eq} = 9.7247739\,\Omega$  
$b = -0.01667269\,V$

In [4]:
import numpy as np

def regressao(x, y, v):
    V = v(x)
    Vt = V.T
    return np.linalg.inv(Vt @ V) @ (Vt @ y)

def main():
    # V = Req * I + b
    I = np.array([0.5, 1.1, 1.5, 2.2, 2.5, 3.1], dtype=float)
    V = np.array([5.1, 10.3, 15.2, 20.1, 24.7, 30.5], dtype=float)

    v = lambda x: np.column_stack((np.ones(len(x)), x))
    A = regressao(I, V, v)
    b = A[0]
    Req = A[1]

    print(f"Req = {Req:.7f} Ohms")
    print(f"b   = {b:.8f} V")

if __name__ == "__main__":
    main()

Req = 9.7247740 Ohms
b   = -0.01667269 V


## Atividade 5

A relação entre tensão, corrente, resistência e indutância é:

$$
V(t) = L \frac{dI(t)}{dt} + R I(t)
$$

onde:  
$R = 5\,\Omega$, $L = 0.1\,H$

| Tempo $t$ (s) | Corrente $I(t)$ (A) |
|----------------|--------------------|
| 0.0 | 0.00 |
| 0.1 | 0.82 |
| 0.2 | 1.36 |
| 0.3 | 1.60 |
| 0.4 | 1.73 |

Sabendo que $I(t)$ cresce linearmente no tempo, estime $V(0.5)$.

**Sugestão:** método da **diferença finita progressiva**.  
**Resposta:** $V(0.5) = 12.29397\,V$

In [5]:
import numpy as np

def regressao(x, y, v):
    V = v(x)
    Vt = V.T
    return np.linalg.inv(Vt @ V) @ (Vt @ y)

def dp(f, x, h):
    return (f(x + h) - f(x)) / h

def main():
    R = 5.0
    L = 0.1
    t = np.array([0.0, 0.1, 0.2, 0.3, 0.4], dtype=float)
    I = np.array([0.00, 0.82, 1.36, 1.60, 1.73], dtype=float)

    # I(t) cresce linearmente no tempo: I(t) = a1 * t + a0
    v = lambda x: np.column_stack((np.ones(len(x)), x))
    A = regressao(t, I, v)
    a0, a1 = A[0], A[1]

    I_func = lambda t_val: a0 + a1 * t_val
    h = 0.0001
    dI_dt = dp(I_func, 0.5, h)
    I_05 = I_func(0.5)

    V_05 = L * dI_dt + R * I_05
    print(f"Ajuste linear: I(t) = {a1:.5f}*t + {a0:.5f}")
    print(f"I(0.5)   = {I_05:.5f} A")
    print(f"dI/dt    = {dI_dt:.5f} A/s")
    print(f"V(0.5)   = {V_05:.5f} V")

if __name__ == "__main__":
    main()

Ajuste linear: I(t) = 4.24000*t + 0.25400
I(0.5)   = 2.37400 A
dI/dt    = 4.24000 A/s
V(0.5)   = 12.29400 V


## Atividade 6

Repita a tarefa anterior, supondo que $I(t)$ cresce quadraticamente no tempo.

**Sugestão:** método da **diferença finita regressiva**.  
**Resposta:** $V(0.5) = 7.37401\,V$

In [6]:
import numpy as np

def regressao(x, y, v):
    V = v(x)
    Vt = V.T
    return np.linalg.inv(Vt @ V) @ (Vt @ y)

def dr(f, x, h):
    return (f(x) - f(x - h)) / h

def main():
    R = 5.0
    L = 0.1
    t = np.array([0.0, 0.1, 0.2, 0.3, 0.4], dtype=float)
    I = np.array([0.00, 0.82, 1.36, 1.60, 1.73], dtype=float)

    # I(t) cresce quadraticamente no tempo: I(t) = a2*t^2 + a1*t + a0
    v = lambda x: np.column_stack((np.ones(len(x)), x, x**2))
    A = regressao(t, I, v)
    a0, a1, a2 = A[0], A[1], A[2]

    I_func = lambda t_val: a0 + a1 * t_val + a2 * (t_val**2)
    h = 0.0001
    dI_dt = dr(I_func, 0.5, h)
    I_05 = I_func(0.5)

    V_05 = L * dI_dt + R * I_05
    print(f"Ajuste quadrático: I(t) = {a2:.5f}*t^2 + {a1:.5f}*t + {a0:.5f}")
    print(f"I(0.5)   = {I_05:.5f} A")
    print(f"dI/dt    = {dI_dt:.5f} A/s")
    print(f"V(0.5)   = {V_05:.5f} V")

if __name__ == "__main__":
    main()

Ajuste quadrático: I(t) = -12.00000*t^2 + 9.04000*t + 0.01400
I(0.5)   = 1.53400 A
dI/dt    = -2.95880 A/s
V(0.5)   = 7.37412 V


## Atividade 7

Repita novamente, considerando $I(t)$ cúbico no tempo.

**Sugestão:** método da **diferença finita central**.  
**Resposta:** $V(0.5) = 8.89841\,V$

In [7]:
import numpy as np

def regressao(x, y, v):
    V = v(x)
    Vt = V.T
    return np.linalg.inv(Vt @ V) @ (Vt @ y)

def dc(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)

def main():
    R = 5.0
    L = 0.1
    t = np.array([0.0, 0.1, 0.2, 0.3, 0.4], dtype=float)
    I = np.array([0.00, 0.82, 1.36, 1.60, 1.73], dtype=float)

    # I(t) cúbico no tempo: I(t) = a3*t^3 + a2*t^2 + a1*t + a0
    v = lambda x: np.column_stack((np.ones(len(x)), x, x**2, x**3))
    A = regressao(t, I, v)
    a0, a1, a2, a3 = A[0], A[1], A[2], A[3]

    I_func = lambda t_val: a0 + a1 * t_val + a2 * (t_val**2) + a3 * (t_val**3)
    h = 0.0001
    dI_dt = dc(I_func, 0.5, h)
    I_05 = I_func(0.5)

    V_05 = L * dI_dt + R * I_05
    print(f"Ajuste cúbico: I(t) = {a3:.5f}*t^3 + {a2:.5f}*t^2 + {a1:.5f}*t + {a0:.5f}")
    print(f"I(0.5)   = {I_05:.5f} A")
    print(f"dI/dt    = {dI_dt:.5f} A/s")
    print(f"V(0.5)   = {V_05:.5f} V")

if __name__ == "__main__":
    main()

Ajuste cúbico: I(t) = 14.16667*t^3 + -20.50000*t^2 + 10.25833*t + -0.00300
I(0.5)   = 1.77200 A
dI/dt    = 0.38333 A/s
V(0.5)   = 8.89833 V


## Atividade 8

Uma empresa de saneamento pública realiza o cálculo de micro drenagem das ruas por meio da pluviosidade das águas de chuva em função do raio da manilha, dado pela fórmula $r$ é:

$$
P(r) = 3\left(1 - \frac{r}{4}\right)^{1/7}
$$

A vazão de água de chuva é dada por:

$$
Q(r) = 2\pi \int_0^r r P(r)\,dr
$$

| Raio $r$ | Resposta $Q(r)$ |
|-----------|----------------|
| 1.25 | 14.235493 |
| 2.55 | 56.396741 |
| 3.15 | 83.297811 |
| 3.95 | 121.398311 |

**Sugestão:** usar a **regra de quadratura pelo ponto médio**.

In [8]:
import numpy as np

def medio(f, a, b):
    h = b - a
    return h * f((a + b) / 2)

def integral(metodo, f, a, b, n=1e-3):
    s = 0.0
    c = a
    d = a + n
    while d <= b + 1e-12:
        s += metodo(f, c, d)
        c = d
        d += n
    return s

def P(r):
    return 3.0 * (1.0 - r / 4.0) ** (1.0 / 7.0)

def integrand(r):
    return 2.0 * np.pi * r * P(r)

def main():
    raios = [1.25, 2.55, 3.15, 3.95]
    print("Vazão Q(r) pela Regra do Ponto Médio:")
    for r in raios:
        q = integral(medio, integrand, 0.0, r, n=1e-3)
        print(f"r = {r:<4} -> Q(r) = {q:.6f}")

if __name__ == "__main__":
    main()

Vazão Q(r) pela Regra do Ponto Médio:
r = 1.25 -> Q(r) = 14.235494
r = 2.55 -> Q(r) = 56.396742
r = 3.15 -> Q(r) = 83.297811
r = 3.95 -> Q(r) = 121.398316


## Atividade 9

Repita o exercício anterior, usando a **regra do trapézio**.

In [9]:
import numpy as np

def trapezio(f, a, b):
    h = b - a
    return h * (0.5 * f(a) + 0.5 * f(b))

def integral(metodo, f, a, b, n=1e-3):
    s = 0.0
    c = a
    d = a + n
    while d <= b + 1e-12:
        s += metodo(f, c, d)
        c = d
        d += n
    return s

def P(r):
    return 3.0 * (1.0 - r / 4.0) ** (1.0 / 7.0)

def integrand(r):
    return 2.0 * np.pi * r * P(r)

def main():
    raios = [1.25, 2.55, 3.15, 3.95]
    print("Vazão Q(r) pela Regra do Trapézio:")
    for r in raios:
        q = integral(trapezio, integrand, 0.0, r, n=1e-3)
        print(f"r = {r:<4} -> Q(r) = {q:.6f}")

if __name__ == "__main__":
    main()

Vazão Q(r) pela Regra do Trapézio:
r = 1.25 -> Q(r) = 14.235493
r = 2.55 -> Q(r) = 56.396741
r = 3.15 -> Q(r) = 83.297810
r = 3.95 -> Q(r) = 121.398300


## Atividade 10

Repita o exercício anterior, usando a **regra de Simpson**.

In [10]:
import numpy as np

def simpson(f, a, b):
    h = (b - a) / 2
    return h * ((1 / 3) * f(a) + (4 / 3) * f((a + b) / 2) + (1 / 3) * f(b))

def integral(metodo, f, a, b, n=1e-3):
    s = 0.0
    c = a
    d = a + n
    while d <= b + 1e-12:
        s += metodo(f, c, d)
        c = d
        d += n
    return s

def P(r):
    return 3.0 * (1.0 - r / 4.0) ** (1.0 / 7.0)

def integrand(r):
    return 2.0 * np.pi * r * P(r)

def main():
    raios = [1.25, 2.55, 3.15, 3.95]
    print("Vazão Q(r) pela Regra de Simpson:")
    for r in raios:
        q = integral(simpson, integrand, 0.0, r, n=1e-3)
        print(f"r = {r:<4} -> Q(r) = {q:.6f}")

if __name__ == "__main__":
    main()

Vazão Q(r) pela Regra de Simpson:
r = 1.25 -> Q(r) = 14.235493
r = 2.55 -> Q(r) = 56.396741
r = 3.15 -> Q(r) = 83.297811
r = 3.95 -> Q(r) = 121.398311


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 13"
git push origin semana13
```